In [1]:
# импортируем функцию для поднятия сессии и для отображения занятой памяти
import sys
import os

from functools import reduce
import yaml
from itertools import chain
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
from dateutil.relativedelta import relativedelta
import datetime
from collections import defaultdict
import pyspark
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import MinMaxScaler

from pyspark.sql import functions as F, types as T, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import MapType, StringType, IntegerType, DoubleType, ByteType, LongType
from pyspark.sql.functions import from_json
from pyspark.sql import SparkSession
from pyspark import SparkConf

from dataclasses import dataclass
from IPython.display import display, clear_output
from typing import List, Union, Callable
import subprocess
import time

from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('../')
sys.path.append('../../')
sys.path.append('../../../')
sys.path.append('../../../../')
# sys.path.insert(0, "/home/datalab/nfs/zaripov/avatar_fm")
sys.path.insert(0, "/home/datalab/nfs/bogachev/")

import pyspark
from pyspark.sql import functions as F, types as T
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import SparkSession
from pyspark import SparkConf
from tools.spark_session import create_spark_session

import time
import datetime

from avatar_fm.avatar.preprocessing.spark.pipeline import TabularPreprocessor

### spark = create_spark_session(app_name='fmlib-feature-selection1', executor_instances=10)

In [2]:
spark = create_spark_session(app_name='fmlib-feature-selection1', executor_instances=10)

Setting spark.hadoop.yarn.resourcemanager.principal to 23919316_omega-sbrf-ru
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Статус: OK
Spark UI: https://ci04139341-prom-datalabpro.apps.prom-terra000035-ias.ocp.ca.sbrf.ru/ci04139341-p-8713-avatar/datalabpro/jserver/proxy/4041/jobs/


In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
train_df = spark.read.parquet("/user/team/team_ai_avatar/ds/zaripov/feature_selection_benchmark/product_name=sa_response_erkc/split_type=train")
valid_df = spark.read.parquet("/user/team/team_ai_avatar/ds/rusakov/feature_selection_benchmark/benchmark_datasets/product_name=tdbase_response/split_type=valid")
test_df = spark.read.parquet("/user/team/team_ai_avatar/ds/rusakov/feature_selection_benchmark/benchmark_datasets/product_name=tdbase_response/split_type=test")
with open("/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/features_bio.yaml", "r") as f:
    feat_cols = yaml.safe_load(f)

ERROR:root:KeyboardInterrupt while sending command.                 (0 + 0) / 1]
Traceback (most recent call last):
  File "/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib64/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
from fmlib.feature_selection import (
    FeatureSchema,
    FeatureSelectionConfig,
    FeatureSelectionPipeline,
)
config = FeatureSelectionConfig.from_yaml(
 
    "/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/configs/feature_selection/psi.yaml"
)

In [ ]:
pipeline = FeatureSelectionPipeline(config)
result = pipeline.fit_select(
    spark=spark,
    datasets={
        "train": train_df,
        # "valid": valid_df,
        # "test": test_df,
    }, 
    schema=FeatureSchema(
        categorical=feat_cols["cat_cols"],
        continuous=feat_cols["num_cols"],
        target="target_attr_1",
        task_type="binary_classification",
        time="month_part",
        # split="month_part",
        fold=None,
        id_columns=feat_cols["id_cols"],
    )
)
result.dropped_features

In [ ]:
len([i for i in result.dropped_features if i.stage == "statistics"])

In [ ]:
len([i for i in result.dropped_features if i.method == "null_rate"])

In [ ]:
len([i for i in result.dropped_features if i.method == "constants"])

In [ ]:
len([i for i in result.dropped_features if i.method == "low_variance"])

In [ ]:
len([i for i in result.dropped_features if i.method == "correlation"])

In [ ]:
len([i for i in result.dropped_features if i.method == "psi"])

In [ ]:
result.dropped_features[-50:]

In [ ]:
result.save("lol.json")

In [ ]:
# (
#     spark.read.parquet(
#         "/user/team/team_ai_avatar/ds/zaripov/feature_selection_benchmark/product_name=sa_response_erkc"
#     )
#     .write
#     .partitionBy("split_type")
#     .mode('overwrite')
#     .format('parquet')
#     .option('compression', 'snappy')
#     .parquet("/user/team/team_ai_avatar/ds/zaripov/feature_selection_benchmark/product_name=sa_response_erkc_buff")
# )

In [ ]:
# (
#     spark.read.parquet(
#         "/user/team/team_ai_avatar/ds/zaripov/feature_selection_benchmark/product_name=sa_response_sms"
#     )
#     .write
#     .partitionBy("split_type")
#     .mode('overwrite')
#     .format('parquet')
#     .option('compression', 'snappy')
#     .parquet("/user/team/team_ai_avatar/ds/zaripov/feature_selection_benchmark/product_name=sa_response_sms_buff")
# )

In [23]:
import yaml

with open("features_bio.yaml", "r") as f:
    bio = yaml.safe_load(f)






with open("/home/datalab/nfs/bogachev/fmlib-feature-selection/dev/bogachev/results/pipeline/SA_SMA/variance_1e-05_minmax_cols.txt", "r") as f:
    old = (f.read()).split("\n")

import json
with open("/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/sms/statistics_low_variance_results.json", "r") as f:
    a = json.load(f)
l1  =  [ i["feature"] for i in a["dropped_features"]]
with open("/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/sms/statistics_constants_results.json", "r") as f:
    a = json.load(f)
l2  =  [ i["feature"] for i in a["dropped_features"]]

new = list(set(l1) - set(l2) )

In [26]:
len([i for i in old if i in bio["cat_cols"]])

204

In [32]:
import yaml

with open("features_bio.yaml", "r") as f:
    bio = yaml.safe_load(f)






with open("/home/datalab/nfs/bogachev/fmlib-feature-selection/dev/bogachev/results/pipeline/PD/variance_1e-05_minmax_cols.txt", "r") as f:
    old_1 = (f.read()).split("\n")


with open("/home/datalab/nfs/bogachev/fmlib-feature-selection/dev/bogachev/results/sequential/PD/constant_0.95_cols.txt", "r") as f:
    old_2 = (f.read()).split("\n")
old  = list(set(old_1) - set(old_2) )
import json
with open("/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/sms/statistics_low_variance_results.json", "r") as f:
    a = json.load(f)
l1  =  [ i["feature"] for i in a["dropped_features"]]
with open("/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/sms/statistics_constants_results.json", "r") as f:
    a = json.load(f)
l2  =  [ i["feature"] for i in a["dropped_features"]]

new = list(set(l1) - set(l2) )

len([i for i in old if i in bio["cat_cols"]])

20

In [54]:
import json
with open("/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/sms/statistics_psi_results.json") as f:
    f = json.load(f)
ar =  [ i["feature"] for i in f["dropped_features"]]

with open("untitled.txt","w") as f:
    for i in ar:
        f.write(i+"\n")

In [53]:
import json
with open("/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/sms/statistics_correlation_results.json") as f:
    f = json.load(f)
ar =  [ i["feature"] for i in f["dropped_features"]]

with open("untitled.txt","w") as f:
    for i in ar:
        f.write(i+"\n")

In [ ]:
w